<a href="https://colab.research.google.com/github/sway4em/566-cricket-colab/blob/new/pipeline_new_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building the Best IPL Team
## Player Archetype Discovery via Unsupervised Learning

**Train:** 2008 - 2024 | **Test:** 2025 - 2026

### Setup
Upload your `data/` folder to Google Drive. Update `DATA_DIR` below to point to it.

In [ ]:
# Mount Google Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q umap-learn

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from scipy import stats

import umap
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

# UPDATE THIS PATH to where you uploaded the data/ folder in Google Drive
DATA_DIR = Path('/content/drive/MyDrive/566-term-project/data')

# Output directory for saved figures
OUTPUT_DIR = Path('/content/drive/MyDrive/566-term-project/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Phase 1: Load and Combine Data

In [ ]:
csv_files = list(DATA_DIR.glob('*.csv'))
print(f"Found {len(csv_files)} match files")

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    dfs.append(df)

ball_data = pd.concat(dfs, ignore_index=True)
print(f"Total deliveries: {len(ball_data):,}")
print(f"Seasons (raw): {ball_data['season'].unique()}")
ball_data.head()

In [ ]:
# Normalize season labels
season_map = {
    '2007/08': '2008',
    '2009/10': '2010',
    '2020/21': '2020',
}
ball_data['season'] = ball_data['season'].astype(str).replace(season_map)
ball_data['season'] = ball_data['season'].astype(int)
print(f"Seasons after normalization: {sorted(ball_data['season'].unique())}")

In [ ]:
# Train/test split by season
TEST_SEASONS = [2025, 2026]
train_data = ball_data[~ball_data['season'].isin(TEST_SEASONS)].copy()
test_data = ball_data[ball_data['season'].isin(TEST_SEASONS)].copy()

print(f"Train: {len(train_data):,} deliveries ({train_data['season'].min()}-{train_data['season'].max()})")
print(f"Test:  {len(test_data):,} deliveries (seasons {TEST_SEASONS})")

## Phase 2: Feature Engineering

Compute per-player, per-season feature vectors. We define three game phases:
- **Powerplay:** Overs 1-6
- **Middle:** Overs 7-15
- **Death:** Overs 16-20

In [ ]:
def get_phase(ball_col):
    """Map ball number (e.g. 3.4 means over 4, ball 4) to game phase."""
    over = ball_col.astype(str).str.split('.').str[0].astype(int)
    phase = pd.Series('middle', index=ball_col.index)
    phase[over < 6] = 'powerplay'
    phase[over >= 15] = 'death'
    return phase

train_data['phase'] = get_phase(train_data['ball'])
test_data['phase'] = get_phase(test_data['ball'])

train_data['phase'].value_counts()

In [ ]:
def compute_batting_features(data):
    """Compute per-player, per-season batting features."""
    # Each row where the player is the striker counts as a ball faced
    # (unless it's a wide, which doesn't count as a ball faced)
    batting = data[data['wides'].isna() | (data['wides'] == 0)].copy()
    batting['is_boundary'] = batting['runs_off_bat'].isin([4, 6]).astype(int)
    batting['is_six'] = (batting['runs_off_bat'] == 6).astype(int)
    batting['is_dot'] = (batting['runs_off_bat'] == 0).astype(int)
    batting['is_dismissed'] = batting['player_dismissed'].notna().astype(int)

    group_cols = ['striker', 'season']

    # Overall stats
    overall = batting.groupby(group_cols).agg(
        total_runs=('runs_off_bat', 'sum'),
        balls_faced=('runs_off_bat', 'count'),
        boundaries=('is_boundary', 'sum'),
        sixes=('is_six', 'sum'),
        dots=('is_dot', 'sum'),
        dismissals=('is_dismissed', 'sum'),
    ).reset_index()

    overall['strike_rate'] = (overall['total_runs'] / overall['balls_faced']) * 100
    overall['boundary_pct'] = overall['boundaries'] / overall['balls_faced']
    overall['six_pct'] = overall['sixes'] / overall['balls_faced']
    overall['dot_pct'] = overall['dots'] / overall['balls_faced']
    overall['average'] = overall['total_runs'] / overall['dismissals'].clip(lower=1)

    # Phase-wise strike rates
    phase_sr = batting.groupby(group_cols + ['phase']).agg(
        phase_runs=('runs_off_bat', 'sum'),
        phase_balls=('runs_off_bat', 'count'),
        phase_dots=('is_dot', 'sum'),
        phase_boundaries=('is_boundary', 'sum'),
    ).reset_index()

    phase_sr['phase_sr'] = (phase_sr['phase_runs'] / phase_sr['phase_balls']) * 100
    phase_sr['phase_dot_pct'] = phase_sr['phase_dots'] / phase_sr['phase_balls']
    phase_sr['phase_boundary_pct'] = phase_sr['phase_boundaries'] / phase_sr['phase_balls']

    # Pivot phases into columns
    for metric in ['phase_sr', 'phase_dot_pct', 'phase_boundary_pct']:
        pivot = phase_sr.pivot_table(
            index=group_cols, columns='phase', values=metric
        ).reset_index()
        pivot.columns = [f'bat_{metric}_{c}' if c in ['powerplay', 'middle', 'death'] else c
                        for c in pivot.columns]
        overall = overall.merge(pivot, on=group_cols, how='left')

    # Consistency: std of runs per innings
    innings_runs = batting.groupby(['striker', 'season', 'match_id', 'innings']).agg(
        innings_runs=('runs_off_bat', 'sum'),
        innings_balls=('runs_off_bat', 'count'),
    ).reset_index()

    consistency = innings_runs.groupby(group_cols).agg(
        runs_std=('innings_runs', 'std'),
        avg_balls_per_innings=('innings_balls', 'mean'),
        num_innings=('innings_runs', 'count'),
    ).reset_index()

    overall = overall.merge(consistency, on=group_cols, how='left')

    # Acceleration: death SR / powerplay SR
    pp_col = 'bat_phase_sr_powerplay'
    death_col = 'bat_phase_sr_death'
    if pp_col in overall.columns and death_col in overall.columns:
        overall['acceleration'] = overall[death_col] / overall[pp_col].clip(lower=1)

    overall = overall.rename(columns={'striker': 'player'})
    return overall

bat_train = compute_batting_features(train_data)
bat_test = compute_batting_features(test_data)
print(f"Batting features: {bat_train.shape}")
bat_train.head()

In [ ]:
def compute_bowling_features(data):
    """Compute per-player, per-season bowling features."""
    bowling = data.copy()
    # Legitimate deliveries (not wides/noballs) for balls bowled count
    bowling['is_legit_delivery'] = (
        (bowling['wides'].isna() | (bowling['wides'] == 0)) &
        (bowling['noballs'].isna() | (bowling['noballs'] == 0))
    ).astype(int)
    bowling['runs_conceded'] = bowling['runs_off_bat'] + bowling['wides'].fillna(0) + bowling['noballs'].fillna(0)
    bowling['is_dot'] = ((bowling['runs_off_bat'] == 0) & (bowling['extras'].fillna(0) == 0)).astype(int)
    bowling['is_boundary_conceded'] = bowling['runs_off_bat'].isin([4, 6]).astype(int)
    bowling['is_wicket'] = (bowling['wicket_type'].notna() &
                            ~bowling['wicket_type'].isin(['run out', 'retired hurt', 'obstructing the field'])).astype(int)

    group_cols = ['bowler', 'season']

    overall = bowling.groupby(group_cols).agg(
        total_runs_conceded=('runs_conceded', 'sum'),
        balls_bowled=('is_legit_delivery', 'sum'),
        total_deliveries=('bowler', 'count'),
        wickets=('is_wicket', 'sum'),
        dots_bowled=('is_dot', 'sum'),
        boundaries_conceded=('is_boundary_conceded', 'sum'),
        extras_given=('extras', 'sum'),
    ).reset_index()

    overall['economy'] = (overall['total_runs_conceded'] / overall['balls_bowled']) * 6
    overall['bowling_sr'] = overall['balls_bowled'] / overall['wickets'].clip(lower=1)
    overall['dot_pct_bowl'] = overall['dots_bowled'] / overall['total_deliveries']
    overall['boundary_concede_pct'] = overall['boundaries_conceded'] / overall['total_deliveries']
    overall['extras_rate'] = overall['extras_given'] / overall['total_deliveries']

    # Phase-wise bowling
    phase_bowl = bowling.groupby(group_cols + ['phase']).agg(
        phase_runs_conceded=('runs_conceded', 'sum'),
        phase_balls=('is_legit_delivery', 'sum'),
        phase_wickets=('is_wicket', 'sum'),
        phase_dots=('is_dot', 'sum'),
        phase_deliveries=('bowler', 'count'),
    ).reset_index()

    phase_bowl['phase_economy'] = (phase_bowl['phase_runs_conceded'] / phase_bowl['phase_balls'].clip(lower=1)) * 6
    phase_bowl['phase_dot_pct'] = phase_bowl['phase_dots'] / phase_bowl['phase_deliveries']
    phase_bowl['phase_wicket_rate'] = phase_bowl['phase_wickets'] / phase_bowl['phase_balls'].clip(lower=1)

    for metric in ['phase_economy', 'phase_dot_pct', 'phase_wicket_rate']:
        pivot = phase_bowl.pivot_table(
            index=group_cols, columns='phase', values=metric
        ).reset_index()
        pivot.columns = [f'bowl_{metric}_{c}' if c in ['powerplay', 'middle', 'death'] else c
                        for c in pivot.columns]
        overall = overall.merge(pivot, on=group_cols, how='left')

    # Wicket type distribution
    wicket_types = bowling[bowling['is_wicket'] == 1].groupby(group_cols)['wicket_type'].value_counts().unstack(fill_value=0)
    if not wicket_types.empty:
        wicket_types = wicket_types.div(wicket_types.sum(axis=1), axis=0)
        wicket_types.columns = [f'wkt_pct_{c}' for c in wicket_types.columns]
        wicket_types = wicket_types.reset_index()
        overall = overall.merge(wicket_types, on=group_cols, how='left')

    overall = overall.rename(columns={'bowler': 'player'})
    return overall

bowl_train = compute_bowling_features(train_data)
bowl_test = compute_bowling_features(test_data)
print(f"Bowling features: {bowl_train.shape}")
bowl_train.head()

In [ ]:
def compute_batting_position_features(data):
    """Infer batting position from when a player first appears in each innings."""
    # Number deliveries sequentially within each innings
    data_sorted = data.sort_values(['match_id', 'innings', 'ball']).copy()
    data_sorted['delivery_num'] = data_sorted.groupby(['match_id', 'innings']).cumcount()

    # For each batter, find the delivery number they first appeared at in each innings
    first_appearance = data_sorted.groupby(['match_id', 'innings', 'striker'])['delivery_num'].min().reset_index()
    first_appearance.columns = ['match_id', 'innings', 'player', 'entry_delivery']

    # Merge season info
    match_seasons = data_sorted[['match_id', 'season']].drop_duplicates()
    first_appearance = first_appearance.merge(match_seasons, on='match_id')

    # Aggregate per player-season
    group_cols = ['player', 'season']
    position_features = first_appearance.groupby(group_cols).agg(
        avg_entry_delivery=('entry_delivery', 'mean'),
        median_entry_delivery=('entry_delivery', 'median'),
        entry_std=('entry_delivery', 'std'),
    ).reset_index()

    # What % of innings does the player open (entry at delivery 0)?
    opens = first_appearance[first_appearance['entry_delivery'] == 0].groupby(group_cols).size().reset_index(name='times_opened')
    total_innings = first_appearance.groupby(group_cols).size().reset_index(name='total_innings_pos')
    position_features = position_features.merge(opens, on=group_cols, how='left')
    position_features = position_features.merge(total_innings, on=group_cols, how='left')
    position_features['times_opened'] = position_features['times_opened'].fillna(0)
    position_features['opener_pct'] = position_features['times_opened'] / position_features['total_innings_pos']

    position_features = position_features.drop(columns=['times_opened', 'total_innings_pos'])
    return position_features

pos_train = compute_batting_position_features(train_data)
pos_test = compute_batting_position_features(test_data)

# Merge into batting features
bat_train = bat_train.merge(pos_train, on=['player', 'season'], how='left')
bat_test = bat_test.merge(pos_test, on=['player', 'season'], how='left')

print(f"Batting features with position: {bat_train.shape}")
# Sanity check
for p in ['CH Gayle', 'V Kohli', 'MS Dhoni', 'JJ Bumrah']:
    row = bat_train[bat_train['player'] == p]
    if len(row) > 0:
        print(f"  {p}: avg entry={row['avg_entry_delivery'].mean():.1f}, opener%={row['opener_pct'].mean():.1%}")

In [ ]:
def compute_bowling_phase_preference(data):
    """What % of a bowler's deliveries are in each phase? Captures role (PP specialist vs death bowler)."""
    phase_dist = data.groupby(['bowler', 'season', 'phase']).size().reset_index(name='deliveries')
    total = data.groupby(['bowler', 'season']).size().reset_index(name='total_deliveries')
    phase_dist = phase_dist.merge(total, on=['bowler', 'season'])
    phase_dist['phase_pct'] = phase_dist['deliveries'] / phase_dist['total_deliveries']

    pivot = phase_dist.pivot_table(index=['bowler', 'season'], columns='phase', values='phase_pct', fill_value=0).reset_index()
    pivot.columns = ['player', 'season', 'bowl_pct_death', 'bowl_pct_middle', 'bowl_pct_powerplay']
    return pivot

bowl_phase_train = compute_bowling_phase_preference(train_data)
bowl_phase_test = compute_bowling_phase_preference(test_data)

# Merge into bowling features
bowl_train = bowl_train.merge(bowl_phase_train, on=['player', 'season'], how='left')
bowl_test = bowl_test.merge(bowl_phase_test, on=['player', 'season'], how='left')

print(f"Bowling features with phase preference: {bowl_train.shape}")
for p in ['JJ Bumrah', 'R Ashwin', 'Rashid Khan']:
    row = bowl_train[bowl_train['player'] == p]
    if len(row) > 0:
        print(f"  {p}: PP={row['bowl_pct_powerplay'].mean():.0%}, Mid={row['bowl_pct_middle'].mean():.0%}, Death={row['bowl_pct_death'].mean():.0%}")

In [ ]:
# Merge batting and bowling features per player-season
# Players who both bat and bowl get full vectors; pure batters/bowlers get NaN for the other side

def merge_features(bat_df, bowl_df, min_balls_bat=30, min_balls_bowl=30):
    """Merge batting and bowling features with minimum threshold filtering."""
    bat_filtered = bat_df[bat_df['balls_faced'] >= min_balls_bat].copy()
    bowl_filtered = bowl_df[bowl_df['balls_bowled'] >= min_balls_bowl].copy()

    # Prefix columns to avoid collision
    bat_cols = [c for c in bat_filtered.columns if c not in ['player', 'season']]
    bowl_cols = [c for c in bowl_filtered.columns if c not in ['player', 'season']]

    bat_filtered = bat_filtered.rename(columns={c: f'bat_{c}' if not c.startswith('bat_') else c for c in bat_cols})
    bowl_filtered = bowl_filtered.rename(columns={c: f'bowl_{c}' if not c.startswith(('bowl_', 'wkt_')) else c for c in bowl_cols})

    merged = bat_filtered.merge(bowl_filtered, on=['player', 'season'], how='outer')

    # Label player type
    has_bat = merged['bat_balls_faced'].notna()
    has_bowl = merged['bowl_balls_bowled'].notna()
    merged['player_type'] = 'unknown'
    merged.loc[has_bat & ~has_bowl, 'player_type'] = 'batter'
    merged.loc[~has_bat & has_bowl, 'player_type'] = 'bowler'
    merged.loc[has_bat & has_bowl, 'player_type'] = 'allrounder'

    print(f"Player-seasons: {len(merged)}")
    print(merged['player_type'].value_counts())
    return merged

features_train = merge_features(bat_train, bowl_train)
features_test = merge_features(bat_test, bowl_test)
features_train.head()

## Phase 3: Separate Dimensionality Reduction

We cluster **batters and bowlers separately** to discover meaningful subtypes within each role
(e.g., openers vs finishers, death bowlers vs middle-overs spinners) rather than the trivial
batter/bowler/allrounder split.

All-rounders get **dual labels** — one batting archetype and one bowling archetype.

In [ ]:
# Separate feature sets for batters and bowlers
bat_feature_cols = [c for c in features_train.columns
                    if c.startswith('bat_') and features_train[c].dtype in ['float64', 'int64']]
# Add position features
bat_feature_cols += [c for c in ['acceleration'] if c in features_train.columns]

bowl_feature_cols = [c for c in features_train.columns
                     if (c.startswith('bowl_') or c.startswith('wkt_')) and features_train[c].dtype in ['float64', 'int64']]

print(f"Batting features ({len(bat_feature_cols)}): {bat_feature_cols}")
print(f"\nBowling features ({len(bowl_feature_cols)}): {bowl_feature_cols}")

# Batters = players with bat data (batters + allrounders)
bat_mask_train = features_train['player_type'].isin(['batter', 'allrounder'])
bat_mask_test = features_test['player_type'].isin(['batter', 'allrounder'])

# Bowlers = players with bowl data (bowlers + allrounders)
bowl_mask_train = features_train['player_type'].isin(['bowler', 'allrounder'])
bowl_mask_test = features_test['player_type'].isin(['bowler', 'allrounder'])

print(f"\nBatters (train): {bat_mask_train.sum()}")
print(f"Bowlers (train): {bowl_mask_train.sum()}")

In [ ]:
# Standardize and PCA for BATTERS
X_bat_train = features_train.loc[bat_mask_train, bat_feature_cols].fillna(0).values
X_bat_test = features_test.loc[bat_mask_test, bat_feature_cols].fillna(0).values

bat_scaler = StandardScaler()
X_bat_train_scaled = bat_scaler.fit_transform(X_bat_train)
X_bat_test_scaled = bat_scaler.transform(X_bat_test)

bat_pca = PCA(n_components=min(15, X_bat_train_scaled.shape[1]))
X_bat_train_pca = bat_pca.fit_transform(X_bat_train_scaled)
X_bat_test_pca = bat_pca.transform(X_bat_test_scaled)

bat_cumvar = np.cumsum(bat_pca.explained_variance_ratio_)
bat_n_components = np.argmax(bat_cumvar >= 0.9) + 1
print(f"Batting PCA: {bat_n_components} components for 90% variance")

# Standardize and PCA for BOWLERS
X_bowl_train = features_train.loc[bowl_mask_train, bowl_feature_cols].fillna(0).values
X_bowl_test = features_test.loc[bowl_mask_test, bowl_feature_cols].fillna(0).values

bowl_scaler = StandardScaler()
X_bowl_train_scaled = bowl_scaler.fit_transform(X_bowl_train)
X_bowl_test_scaled = bowl_scaler.transform(X_bowl_test)

bowl_pca = PCA(n_components=min(15, X_bowl_train_scaled.shape[1]))
X_bowl_train_pca = bowl_pca.fit_transform(X_bowl_train_scaled)
X_bowl_test_pca = bowl_pca.transform(X_bowl_test_scaled)

bowl_cumvar = np.cumsum(bowl_pca.explained_variance_ratio_)
bowl_n_components = np.argmax(bowl_cumvar >= 0.9) + 1
print(f"Bowling PCA: {bowl_n_components} components for 90% variance")

In [ ]:
# PCA variance plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(bat_pca.explained_variance_ratio_) + 1), bat_pca.explained_variance_ratio_)
axes[0].axvline(bat_n_components, color='r', linestyle='--', label=f'90% at k={bat_n_components}')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained')
axes[0].set_title('Batting: Scree Plot')
axes[0].legend()

axes[1].bar(range(1, len(bowl_pca.explained_variance_ratio_) + 1), bowl_pca.explained_variance_ratio_)
axes[1].axvline(bowl_n_components, color='r', linestyle='--', label=f'90% at k={bowl_n_components}')
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Variance Explained')
axes[1].set_title('Bowling: Scree Plot')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# UMAP visualization for batters and bowlers
bat_reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.3)
X_bat_train_umap = bat_reducer.fit_transform(X_bat_train_scaled)

bowl_reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.3)
X_bowl_train_umap = bowl_reducer.fit_transform(X_bowl_train_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(X_bat_train_umap[:, 0], X_bat_train_umap[:, 1], alpha=0.4, s=15, c='steelblue')
axes[0].set_title('UMAP: Batters')
axes[0].set_xlabel('UMAP-1')
axes[0].set_ylabel('UMAP-2')

axes[1].scatter(X_bowl_train_umap[:, 0], X_bowl_train_umap[:, 1], alpha=0.4, s=15, c='firebrick')
axes[1].set_title('UMAP: Bowlers')
axes[1].set_xlabel('UMAP-1')
axes[1].set_ylabel('UMAP-2')

plt.tight_layout()
plt.show()

In [ ]:
# Silhouette analysis: find optimal k for BATTING clusters
X_bat_cluster = X_bat_train_pca[:, :bat_n_components]

bat_k_range = range(3, 10)
bat_silhouettes = []
for k in bat_k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_bat_cluster)
    bat_silhouettes.append(silhouette_score(X_bat_cluster, labels))

# Silhouette analysis: find optimal k for BOWLING clusters
X_bowl_cluster = X_bowl_train_pca[:, :bowl_n_components]

bowl_k_range = range(3, 10)
bowl_silhouettes = []
for k in bowl_k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_bowl_cluster)
    bowl_silhouettes.append(silhouette_score(X_bowl_cluster, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(bat_k_range), bat_silhouettes, 'bo-')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_title('Batting: Optimal k')

axes[1].plot(list(bowl_k_range), bowl_silhouettes, 'ro-')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Bowling: Optimal k')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cluster_selection.png', dpi=150, bbox_inches='tight')
plt.show()

bat_best_k = list(bat_k_range)[np.argmax(bat_silhouettes)]
bowl_best_k = list(bowl_k_range)[np.argmax(bowl_silhouettes)]
print(f"Best batting k: {bat_best_k} (silhouette: {max(bat_silhouettes):.3f})")
print(f"Best bowling k: {bowl_best_k} (silhouette: {max(bowl_silhouettes):.3f})")

## Phase 4: Separate Clustering — Batting & Bowling Archetypes

In [ ]:
# Fit K-Means for BATTING archetypes
bat_kmeans = KMeans(n_clusters=bat_best_k, random_state=42, n_init=10)
bat_labels_train = bat_kmeans.fit_predict(X_bat_cluster)

X_bat_test_cluster = X_bat_test_pca[:, :bat_n_components]
bat_labels_test = bat_kmeans.predict(X_bat_test_cluster)

# Assign batting cluster labels
features_train.loc[bat_mask_train, 'bat_cluster'] = bat_labels_train
features_test.loc[bat_mask_test, 'bat_cluster'] = bat_labels_test

# Fit K-Means for BOWLING archetypes
bowl_kmeans = KMeans(n_clusters=bowl_best_k, random_state=42, n_init=10)
bowl_labels_train = bowl_kmeans.fit_predict(X_bowl_cluster)

X_bowl_test_cluster = X_bowl_test_pca[:, :bowl_n_components]
bowl_labels_test = bowl_kmeans.predict(X_bowl_test_cluster)

# Assign bowling cluster labels
features_train.loc[bowl_mask_train, 'bowl_cluster'] = bowl_labels_train
features_test.loc[bowl_mask_test, 'bowl_cluster'] = bowl_labels_test

print("Batting cluster distribution (train):")
print(features_train['bat_cluster'].value_counts().sort_index())
print(f"\nBowling cluster distribution (train):")
print(features_train['bowl_cluster'].value_counts().sort_index())

In [ ]:
# Interpret BATTING clusters
bat_profiles = features_train[bat_mask_train].groupby('bat_cluster')[bat_feature_cols].mean()

key_bat = [c for c in ['bat_strike_rate', 'bat_boundary_pct', 'bat_dot_pct', 'bat_average',
                        'bat_phase_sr_powerplay', 'bat_phase_sr_death', 'acceleration',
                        'bat_avg_entry_delivery', 'bat_opener_pct', 'bat_six_pct'] if c in bat_feature_cols]

print("=== BATTING ARCHETYPE PROFILES ===")
print(bat_profiles[key_bat].round(2))
print()

# Name the clusters based on their profiles
print("\nCluster interpretation guide:")
print("- High opener_pct + high PP SR = Aggressive Opener")
print("- High avg_entry_delivery + high death SR = Finisher")
print("- Low dot_pct + high average + moderate SR = Anchor")
print("- High boundary_pct + high SR across phases = Power Hitter")

In [ ]:
# Interpret BOWLING clusters
bowl_profiles = features_train[bowl_mask_train].groupby('bowl_cluster')[bowl_feature_cols].mean()

key_bowl = [c for c in ['bowl_economy', 'bowl_bowling_sr', 'bowl_dot_pct_bowl',
                         'bowl_boundary_concede_pct', 'bowl_pct_powerplay', 'bowl_pct_middle',
                         'bowl_pct_death', 'bowl_phase_economy_powerplay',
                         'bowl_phase_economy_death', 'bowl_phase_wicket_rate_death'] if c in bowl_feature_cols]

print("=== BOWLING ARCHETYPE PROFILES ===")
print(bowl_profiles[key_bowl].round(2))
print()

print("\nCluster interpretation guide:")
print("- High bowl_pct_death + low economy = Death Specialist")
print("- High bowl_pct_middle + high dot% = Middle-Overs Controller (spinners)")
print("- High bowl_pct_powerplay + high wicket rate = New Ball Striker")
print("- High economy + low dot% = Part-Timer / Occasional Bowler")

In [ ]:
# Visualize clusters in UMAP space
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

scatter1 = axes[0].scatter(X_bat_train_umap[:, 0], X_bat_train_umap[:, 1],
                           c=bat_labels_train, cmap='tab10', alpha=0.5, s=15)
axes[0].set_title(f'Batting Archetypes (k={bat_best_k})')
axes[0].set_xlabel('UMAP-1')
axes[0].set_ylabel('UMAP-2')
plt.colorbar(scatter1, ax=axes[0])

scatter2 = axes[1].scatter(X_bowl_train_umap[:, 0], X_bowl_train_umap[:, 1],
                           c=bowl_labels_train, cmap='tab10', alpha=0.5, s=15)
axes[1].set_title(f'Bowling Archetypes (k={bowl_best_k})')
axes[1].set_xlabel('UMAP-1')
axes[1].set_ylabel('UMAP-2')
plt.colorbar(scatter2, ax=axes[1])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'clusters_umap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Spot-check: known players and their cluster assignments
known_batters = ['V Kohli', 'CH Gayle', 'DA Warner', 'MS Dhoni', 'AB de Villiers',
                 'KL Rahul', 'SA Yadav', 'HH Pandya', 'RG Sharma']
known_bowlers = ['JJ Bumrah', 'Rashid Khan', 'R Ashwin', 'YS Chahal',
                 'DJ Bravo', 'SP Narine', 'B Kumar', 'T Natarajan']

print("=== BATTING ARCHETYPES: Known Players ===")
bat_check = features_train[features_train['player'].isin(known_batters + known_bowlers)]
bat_check = bat_check[bat_check['bat_cluster'].notna()]
# Show most recent season per player
bat_latest = bat_check.sort_values('season').groupby('player').tail(1)
print(bat_latest[['player', 'season', 'bat_cluster', 'player_type']].sort_values('bat_cluster').to_string(index=False))

print("\n=== BOWLING ARCHETYPES: Known Players ===")
bowl_check = features_train[features_train['player'].isin(known_batters + known_bowlers)]
bowl_check = bowl_check[bowl_check['bowl_cluster'].notna()]
bowl_latest = bowl_check.sort_values('season').groupby('player').tail(1)
print(bowl_latest[['player', 'season', 'bowl_cluster', 'player_type']].sort_values('bowl_cluster').to_string(index=False))

In [ ]:
# Create a UNIFIED archetype label for each player-season
# Batters: "BAT-{cluster}", Bowlers: "BOWL-{cluster}", All-rounders: "BAT-{x}_BOWL-{y}"

def assign_archetype(row):
    has_bat = pd.notna(row.get('bat_cluster'))
    has_bowl = pd.notna(row.get('bowl_cluster'))
    if has_bat and has_bowl:
        return f"BAT-{int(row['bat_cluster'])}_BOWL-{int(row['bowl_cluster'])}"
    elif has_bat:
        return f"BAT-{int(row['bat_cluster'])}"
    elif has_bowl:
        return f"BOWL-{int(row['bowl_cluster'])}"
    return 'unknown'

features_train['archetype'] = features_train.apply(assign_archetype, axis=1)
features_test['archetype'] = features_test.apply(assign_archetype, axis=1)

print("Archetype distribution (train):")
print(features_train['archetype'].value_counts().head(20))

## Phase 5: Team Composition Analysis

Compare **batting and bowling archetype distributions** of playoff vs non-playoff teams.
We analyze batting and bowling separately since that's where the meaningful subtypes live.

In [ ]:
# Map players to teams per season
def get_player_teams(data):
    """Get player-team-season mapping from ball data."""
    bat_teams = data[['striker', 'batting_team', 'season']].rename(
        columns={'striker': 'player', 'batting_team': 'team'})
    bowl_teams = data[['bowler', 'bowling_team', 'season']].rename(
        columns={'bowler': 'player', 'bowling_team': 'team'})
    all_teams = pd.concat([bat_teams, bowl_teams]).drop_duplicates()
    player_teams = all_teams.groupby(['player', 'season'])['team'].agg(
        lambda x: x.value_counts().index[0]
    ).reset_index()
    return player_teams

player_teams_train = get_player_teams(train_data)
player_teams_test = get_player_teams(test_data)

features_train = features_train.drop(columns=['team'], errors='ignore')
features_test = features_test.drop(columns=['team'], errors='ignore')
features_train = features_train.merge(player_teams_train, on=['player', 'season'], how='left')
features_test = features_test.merge(player_teams_test, on=['player', 'season'], how='left')

print(f"Teams in training data: {features_train['team'].nunique()}")

In [ ]:
# IPL Playoff teams by season (top 4)
playoff_teams = {
    2008: ['Rajasthan Royals', 'Chennai Super Kings', 'Delhi Daredevils', 'Kings XI Punjab'],
    2009: ['Deccan Chargers', 'Royal Challengers Bangalore', 'Delhi Daredevils', 'Chennai Super Kings'],
    2010: ['Chennai Super Kings', 'Mumbai Indians', 'Royal Challengers Bangalore', 'Deccan Chargers'],
    2011: ['Chennai Super Kings', 'Royal Challengers Bangalore', 'Mumbai Indians', 'Kolkata Knight Riders'],
    2012: ['Kolkata Knight Riders', 'Chennai Super Kings', 'Delhi Daredevils', 'Mumbai Indians'],
    2013: ['Mumbai Indians', 'Chennai Super Kings', 'Rajasthan Royals', 'Sunrisers Hyderabad'],
    2014: ['Kolkata Knight Riders', 'Kings XI Punjab', 'Chennai Super Kings', 'Mumbai Indians'],
    2015: ['Mumbai Indians', 'Chennai Super Kings', 'Royal Challengers Bangalore', 'Rajasthan Royals'],
    2016: ['Sunrisers Hyderabad', 'Royal Challengers Bangalore', 'Gujarat Lions', 'Kolkata Knight Riders'],
    2017: ['Mumbai Indians', 'Rising Pune Supergiant', 'Sunrisers Hyderabad', 'Kolkata Knight Riders'],
    2018: ['Chennai Super Kings', 'Sunrisers Hyderabad', 'Kolkata Knight Riders', 'Rajasthan Royals'],
    2019: ['Mumbai Indians', 'Chennai Super Kings', 'Delhi Capitals', 'Sunrisers Hyderabad'],
    2020: ['Mumbai Indians', 'Delhi Capitals', 'Sunrisers Hyderabad', 'Royal Challengers Bangalore'],
    2021: ['Chennai Super Kings', 'Kolkata Knight Riders', 'Delhi Capitals', 'Royal Challengers Bangalore'],
    2022: ['Gujarat Titans', 'Rajasthan Royals', 'Lucknow Super Giants', 'Royal Challengers Bangalore'],
    2023: ['Chennai Super Kings', 'Gujarat Titans', 'Mumbai Indians', 'Lucknow Super Giants'],
    2024: ['Kolkata Knight Riders', 'Sunrisers Hyderabad', 'Rajasthan Royals', 'Royal Challengers Bengaluru'],
}

def is_playoff(row):
    season = row['season']
    team = row['team']
    if season in playoff_teams:
        return team in playoff_teams[season]
    return None

features_train['is_playoff'] = features_train.apply(is_playoff, axis=1)
print(features_train['is_playoff'].value_counts(dropna=False))

In [ ]:
# Batting archetype composition: playoff vs non-playoff
bat_team_comp = features_train[features_train['bat_cluster'].notna()].groupby(
    ['team', 'season', 'is_playoff'])['bat_cluster'].value_counts(normalize=True
).unstack(fill_value=0).reset_index()
bat_team_comp.columns.name = None

bat_cluster_cols = [c for c in bat_team_comp.columns if isinstance(c, (int, float, np.integer, np.floating))]

bat_po = bat_team_comp[bat_team_comp['is_playoff'] == True][bat_cluster_cols].mean()
bat_npo = bat_team_comp[bat_team_comp['is_playoff'] == False][bat_cluster_cols].mean()

# Bowling archetype composition: playoff vs non-playoff
bowl_team_comp = features_train[features_train['bowl_cluster'].notna()].groupby(
    ['team', 'season', 'is_playoff'])['bowl_cluster'].value_counts(normalize=True
).unstack(fill_value=0).reset_index()
bowl_team_comp.columns.name = None

bowl_cluster_cols = [c for c in bowl_team_comp.columns if isinstance(c, (int, float, np.integer, np.floating))]

bowl_po = bowl_team_comp[bowl_team_comp['is_playoff'] == True][bowl_cluster_cols].mean()
bowl_npo = bowl_team_comp[bowl_team_comp['is_playoff'] == False][bowl_cluster_cols].mean()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(bat_cluster_cols))
width = 0.35
axes[0].bar(x - width/2, bat_po.values, width, label='Playoff', color='green', alpha=0.7)
axes[0].bar(x + width/2, bat_npo.values, width, label='Non-Playoff', color='red', alpha=0.7)
axes[0].set_xlabel('Batting Archetype')
axes[0].set_ylabel('Proportion of Batters')
axes[0].set_title('Batting Archetypes: Playoff vs Non-Playoff')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'BAT-{int(c)}' for c in bat_cluster_cols])
axes[0].legend()

x2 = np.arange(len(bowl_cluster_cols))
axes[1].bar(x2 - width/2, bowl_po.values, width, label='Playoff', color='green', alpha=0.7)
axes[1].bar(x2 + width/2, bowl_npo.values, width, label='Non-Playoff', color='red', alpha=0.7)
axes[1].set_xlabel('Bowling Archetype')
axes[1].set_ylabel('Proportion of Bowlers')
axes[1].set_title('Bowling Archetypes: Playoff vs Non-Playoff')
axes[1].set_xticks(x2)
axes[1].set_xticklabels([f'BOWL-{int(c)}' for c in bowl_cluster_cols])
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'playoff_vs_nonplayoff.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical test: which archetypes differ significantly between playoff and non-playoff?
print("=== BATTING ARCHETYPES: Playoff vs Non-Playoff ===")
print(f"{'Archetype':<12} {'Playoff%':<12} {'Non-PO%':<12} {'t-stat':<10} {'p-value':<10}")
print("-" * 56)
for c in bat_cluster_cols:
    po_vals = bat_team_comp[bat_team_comp['is_playoff'] == True][c]
    npo_vals = bat_team_comp[bat_team_comp['is_playoff'] == False][c]
    t_stat, p_val = stats.ttest_ind(po_vals, npo_vals)
    sig = '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
    print(f"BAT-{int(c):<7} {po_vals.mean():<12.3f} {npo_vals.mean():<12.3f} {t_stat:<10.3f} {p_val:<10.4f} {sig}")

print(f"\n=== BOWLING ARCHETYPES: Playoff vs Non-Playoff ===")
print(f"{'Archetype':<12} {'Playoff%':<12} {'Non-PO%':<12} {'t-stat':<10} {'p-value':<10}")
print("-" * 56)
for c in bowl_cluster_cols:
    po_vals = bowl_team_comp[bowl_team_comp['is_playoff'] == True][c]
    npo_vals = bowl_team_comp[bowl_team_comp['is_playoff'] == False][c]
    t_stat, p_val = stats.ttest_ind(po_vals, npo_vals)
    sig = '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
    print(f"BOWL-{int(c):<6} {po_vals.mean():<12.3f} {npo_vals.mean():<12.3f} {t_stat:<10.3f} {p_val:<10.4f} {sig}")

In [ ]:
# Most representative players per BATTING cluster (closest to centroid)
print("=== BATTING ARCHETYPES: Representative Players ===\n")
bat_train_df = features_train[bat_mask_train].reset_index(drop=True)
for c in range(bat_best_k):
    mask = bat_train_df['bat_cluster'] == c
    cluster_data = bat_train_df[mask]
    centroid = bat_kmeans.cluster_centers_[c]
    cluster_pca = X_bat_cluster[mask.values]
    dists = np.linalg.norm(cluster_pca - centroid, axis=1)
    closest_idx = np.argsort(dists)[:8]
    players = cluster_data.iloc[closest_idx][['player', 'season']].values
    unique_names = []
    for name, season in players:
        if name not in unique_names:
            unique_names.append(name)
        if len(unique_names) >= 5:
            break
    print(f"BAT-{c}: {', '.join(unique_names)}")

print("\n=== BOWLING ARCHETYPES: Representative Players ===\n")
bowl_train_df = features_train[bowl_mask_train].reset_index(drop=True)
for c in range(bowl_best_k):
    mask = bowl_train_df['bowl_cluster'] == c
    cluster_data = bowl_train_df[mask]
    centroid = bowl_kmeans.cluster_centers_[c]
    cluster_pca = X_bowl_cluster[mask.values]
    dists = np.linalg.norm(cluster_pca - centroid, axis=1)
    closest_idx = np.argsort(dists)[:8]
    players = cluster_data.iloc[closest_idx][['player', 'season']].values
    unique_names = []
    for name, season in players:
        if name not in unique_names:
            unique_names.append(name)
        if len(unique_names) >= 5:
            break
    print(f"BOWL-{c}: {', '.join(unique_names)}")

## Phase 6: Replacement Recommender

Find similar players using the **same feature space** (batting PCA for batters, bowling PCA for bowlers).
For all-rounders, we combine both distances.

In [ ]:
def find_replacement(player_name, season=None, top_n=5, exclude_team=None):
    """
    Find the most similar players based on their archetype-specific feature vectors.
    Uses batting PCA for batters, bowling PCA for bowlers, both for all-rounders.
    """
    # Find the player
    player_data = features_train[features_train['player'] == player_name]
    if len(player_data) == 0:
        print(f"Player '{player_name}' not found in training data.")
        return None

    if season is None:
        season = player_data['season'].max()

    target = player_data[player_data['season'] == season]
    if len(target) == 0:
        season = player_data['season'].max()
        target = player_data[player_data['season'] == season]

    target = target.iloc[0]
    ptype = target['player_type']

    # Get latest season per candidate
    latest = features_train.groupby('player')['season'].max().reset_index(name='latest_season')
    candidates = features_train.merge(latest, on='player')
    candidates = candidates[candidates['season'] == candidates['latest_season']]
    candidates = candidates[candidates['player'] != player_name]
    if exclude_team:
        candidates = candidates[candidates['team'] != exclude_team]

    # Compute distance based on player type
    if ptype in ['batter', 'allrounder']:
        target_bat_idx = features_train[
            (features_train['player'] == player_name) & (features_train['season'] == season)
        ].index[0]
        target_bat_loc = features_train.index.get_loc(target_bat_idx)

        # Only consider candidates who bat
        cand_bat = candidates[candidates['bat_cluster'].notna()]
        if ptype == 'batter':
            cand_bat = cand_bat[cand_bat['player_type'] == 'batter']

        bat_target_vec = X_bat_train_scaled[np.where(bat_mask_train.values)[0] ==
                                             target_bat_loc][0] if target_bat_loc in np.where(bat_mask_train.values)[0] else None

    if ptype in ['bowler', 'allrounder']:
        target_bowl_idx = features_train[
            (features_train['player'] == player_name) & (features_train['season'] == season)
        ].index[0]
        target_bowl_loc = features_train.index.get_loc(target_bowl_idx)

    # Simpler approach: use the feature columns directly
    if ptype == 'batter':
        target_vec = features_train.loc[
            (features_train['player'] == player_name) & (features_train['season'] == season),
            bat_feature_cols
        ].fillna(0).values[0]
        cand_vecs = candidates[candidates['player_type'].isin(['batter', 'allrounder'])][bat_feature_cols].fillna(0).values
        cand_df = candidates[candidates['player_type'].isin(['batter', 'allrounder'])].copy()
    elif ptype == 'bowler':
        target_vec = features_train.loc[
            (features_train['player'] == player_name) & (features_train['season'] == season),
            bowl_feature_cols
        ].fillna(0).values[0]
        cand_vecs = candidates[candidates['player_type'].isin(['bowler', 'allrounder'])][bowl_feature_cols].fillna(0).values
        cand_df = candidates[candidates['player_type'].isin(['bowler', 'allrounder'])].copy()
    else:  # allrounder
        target_bat = features_train.loc[
            (features_train['player'] == player_name) & (features_train['season'] == season),
            bat_feature_cols
        ].fillna(0).values[0]
        target_bowl = features_train.loc[
            (features_train['player'] == player_name) & (features_train['season'] == season),
            bowl_feature_cols
        ].fillna(0).values[0]
        target_vec = np.concatenate([target_bat, target_bowl])
        cand_df = candidates[candidates['player_type'] == 'allrounder'].copy()
        cand_vecs = np.hstack([
            cand_df[bat_feature_cols].fillna(0).values,
            cand_df[bowl_feature_cols].fillna(0).values
        ])

    if len(cand_df) == 0:
        print("No candidates found.")
        return None

    # Standardize for distance computation
    from sklearn.preprocessing import normalize
    all_vecs = np.vstack([target_vec.reshape(1, -1), cand_vecs])
    scl = StandardScaler()
    all_scaled = scl.fit_transform(all_vecs)
    target_scaled = all_scaled[0]
    cand_scaled = all_scaled[1:]

    distances = np.linalg.norm(cand_scaled - target_scaled, axis=1)
    cand_df = cand_df.copy()
    cand_df['distance'] = distances

    result = cand_df.nsmallest(top_n, 'distance')[
        ['player', 'season', 'team', 'player_type', 'archetype', 'distance']
    ]
    print(f"\nTop {top_n} replacements for {player_name} ({ptype}, season {season}):")
    print(result.to_string(index=False))
    return result

find_replacement('V Kohli', 2024)

In [ ]:
find_replacement('JJ Bumrah', 2024)
print()
find_replacement('RA Jadeja', 2024)

## Phase 7: Validation on Test Set (2025-2026)

Check cluster stability and whether team composition patterns generalize.

In [ ]:
# Cluster stability: do players keep the same archetype across train -> test?
train_latest = features_train.sort_values('season').groupby('player').tail(1)

# Batting stability
bat_stability = features_test[features_test['bat_cluster'].notna()][['player', 'bat_cluster']].merge(
    train_latest[train_latest['bat_cluster'].notna()][['player', 'bat_cluster']],
    on='player', suffixes=('_test', '_train')
)
bat_stability['same'] = bat_stability['bat_cluster_test'] == bat_stability['bat_cluster_train']
bat_stab_pct = bat_stability['same'].mean()

# Bowling stability
bowl_stability = features_test[features_test['bowl_cluster'].notna()][['player', 'bowl_cluster']].merge(
    train_latest[train_latest['bowl_cluster'].notna()][['player', 'bowl_cluster']],
    on='player', suffixes=('_test', '_train')
)
bowl_stability['same'] = bowl_stability['bowl_cluster_test'] == bowl_stability['bowl_cluster_train']
bowl_stab_pct = bowl_stability['same'].mean()

print(f"Batting cluster stability: {bat_stab_pct:.1%} ({len(bat_stability)} players)")
print(f"Bowling cluster stability: {bowl_stab_pct:.1%} ({len(bowl_stability)} players)")

print("\nBatting transition matrix:")
print(pd.crosstab(bat_stability['bat_cluster_train'], bat_stability['bat_cluster_test'], margins=True))
print("\nBowling transition matrix:")
print(pd.crosstab(bowl_stability['bowl_cluster_train'], bowl_stability['bowl_cluster_test'], margins=True))

In [ ]:
# Validate team composition patterns on test seasons (2025/2026)
playoff_teams_test = {
    2025: ['Mumbai Indians', 'Gujarat Titans', 'Punjab Kings', 'Royal Challengers Bengaluru'],
    2026: ['Gujarat Titans', 'Royal Challengers Bengaluru', 'Rajasthan Royals', 'Sunrisers Hyderabad'],
}

features_test['is_playoff'] = features_test.apply(
    lambda r: r['team'] in playoff_teams_test.get(r['season'], []), axis=1)

# Test batting composition
test_bat_comp = features_test[features_test['bat_cluster'].notna()].groupby(
    ['team', 'season', 'is_playoff'])['bat_cluster'].value_counts(normalize=True
).unstack(fill_value=0).reset_index()
test_bat_comp.columns.name = None
test_bat_cols = [c for c in test_bat_comp.columns if isinstance(c, (int, float, np.integer, np.floating))]

test_bat_po = test_bat_comp[test_bat_comp['is_playoff'] == True][test_bat_cols].mean()
test_bat_npo = test_bat_comp[test_bat_comp['is_playoff'] == False][test_bat_cols].mean()

# Plot train vs test patterns side by side
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Train batting
x = np.arange(len(bat_cluster_cols))
axes[0, 0].bar(x - 0.175, bat_po.values, 0.35, label='Playoff', color='green', alpha=0.7)
axes[0, 0].bar(x + 0.175, bat_npo.values, 0.35, label='Non-Playoff', color='red', alpha=0.7)
axes[0, 0].set_title('Train: Batting Archetypes')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels([f'BAT-{int(c)}' for c in bat_cluster_cols])
axes[0, 0].legend()

# Test batting
x2 = np.arange(len(test_bat_cols))
axes[0, 1].bar(x2 - 0.175, test_bat_po.values, 0.35, label='Playoff', color='green', alpha=0.7)
axes[0, 1].bar(x2 + 0.175, test_bat_npo.values, 0.35, label='Non-Playoff', color='red', alpha=0.7)
axes[0, 1].set_title('Test (2025-26): Batting Archetypes')
axes[0, 1].set_xticks(x2)
axes[0, 1].set_xticklabels([f'BAT-{int(c)}' for c in test_bat_cols])
axes[0, 1].legend()

# Train bowling
test_bowl_comp = features_test[features_test['bowl_cluster'].notna()].groupby(
    ['team', 'season', 'is_playoff'])['bowl_cluster'].value_counts(normalize=True
).unstack(fill_value=0).reset_index()
test_bowl_comp.columns.name = None
test_bowl_cols = [c for c in test_bowl_comp.columns if isinstance(c, (int, float, np.integer, np.floating))]

test_bowl_po = test_bowl_comp[test_bowl_comp['is_playoff'] == True][test_bowl_cols].mean()
test_bowl_npo = test_bowl_comp[test_bowl_comp['is_playoff'] == False][test_bowl_cols].mean()

x3 = np.arange(len(bowl_cluster_cols))
axes[1, 0].bar(x3 - 0.175, bowl_po.values, 0.35, label='Playoff', color='green', alpha=0.7)
axes[1, 0].bar(x3 + 0.175, bowl_npo.values, 0.35, label='Non-Playoff', color='red', alpha=0.7)
axes[1, 0].set_title('Train: Bowling Archetypes')
axes[1, 0].set_xticks(x3)
axes[1, 0].set_xticklabels([f'BOWL-{int(c)}' for c in bowl_cluster_cols])
axes[1, 0].legend()

# Test bowling
x4 = np.arange(len(test_bowl_cols))
axes[1, 1].bar(x4 - 0.175, test_bowl_po.values, 0.35, label='Playoff', color='green', alpha=0.7)
axes[1, 1].bar(x4 + 0.175, test_bowl_npo.values, 0.35, label='Non-Playoff', color='red', alpha=0.7)
axes[1, 1].set_title('Test (2025-26): Bowling Archetypes')
axes[1, 1].set_xticks(x4)
axes[1, 1].set_xticklabels([f'BOWL-{int(c)}' for c in test_bowl_cols])
axes[1, 1].legend()

plt.suptitle('Do Playoff Team Composition Patterns Generalize?', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'train_vs_test_validation.png', dpi=150, bbox_inches='tight')
plt.show()

## Phase 8: Structural Validation

1. **Diversity**: Do teams spread players across archetypes (not stack one type)?
2. **Universal structure**: Do all teams have a similar archetype skeleton?
3. **Radar charts**: What does each archetype "look like"?
4. **PCA loadings**: Which features drive the separation?

In [ ]:
# Within-team diversity: Shannon entropy of archetype distributions
from scipy.stats import entropy

def bat_diversity(group):
    counts = group['bat_cluster'].dropna().value_counts(normalize=True)
    return entropy(counts) if len(counts) > 0 else 0

def bowl_diversity(group):
    counts = group['bowl_cluster'].dropna().value_counts(normalize=True)
    return entropy(counts) if len(counts) > 0 else 0

team_bat_entropy = features_train.groupby(['team', 'season']).apply(bat_diversity).reset_index(name='bat_diversity')
team_bowl_entropy = features_train.groupby(['team', 'season']).apply(bowl_diversity).reset_index(name='bowl_diversity')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(team_bat_entropy['bat_diversity'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(team_bat_entropy['bat_diversity'].mean(), color='red', linestyle='--',
                label=f"Mean: {team_bat_entropy['bat_diversity'].mean():.2f}")
axes[0].axvline(np.log(bat_best_k), color='green', linestyle='--',
                label=f"Max (uniform): {np.log(bat_best_k):.2f}")
axes[0].set_xlabel('Shannon Entropy')
axes[0].set_title('Batting Archetype Diversity per Team')
axes[0].legend()

axes[1].hist(team_bowl_entropy['bowl_diversity'], bins=20, edgecolor='black', alpha=0.7, color='firebrick')
axes[1].axvline(team_bowl_entropy['bowl_diversity'].mean(), color='red', linestyle='--',
                label=f"Mean: {team_bowl_entropy['bowl_diversity'].mean():.2f}")
axes[1].axvline(np.log(bowl_best_k), color='green', linestyle='--',
                label=f"Max (uniform): {np.log(bowl_best_k):.2f}")
axes[1].set_xlabel('Shannon Entropy')
axes[1].set_title('Bowling Archetype Diversity per Team')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'team_diversity.png', dpi=150, bbox_inches='tight')
plt.show()

print("High entropy = team fills all archetype roles (diverse squad)")
print("Close to max = no archetype stacking")

In [ ]:
# Team archetype heatmaps for a recent season
recent_season = 2024
recent = features_train[features_train['season'] == recent_season]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Batting heatmap
bat_matrix = recent[recent['bat_cluster'].notna()].groupby(['team', 'bat_cluster']).size().unstack(fill_value=0)
sns.heatmap(bat_matrix, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f'Batting Archetypes by Team ({recent_season})')
axes[0].set_xlabel('Batting Archetype')

# Bowling heatmap
bowl_matrix = recent[recent['bowl_cluster'].notna()].groupby(['team', 'bowl_cluster']).size().unstack(fill_value=0)
sns.heatmap(bowl_matrix, annot=True, fmt='d', cmap='Reds', ax=axes[1])
axes[1].set_title(f'Bowling Archetypes by Team ({recent_season})')
axes[1].set_xlabel('Bowling Archetype')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'team_archetype_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Similar rows across teams = universal team structure (all teams need similar roles)")

In [ ]:
# Radar charts for batting and bowling archetypes
radar_bat_features = [c for c in ['bat_strike_rate', 'bat_boundary_pct', 'bat_dot_pct',
                                   'bat_average', 'bat_avg_entry_delivery', 'bat_opener_pct',
                                   'bat_phase_sr_death', 'acceleration'] if c in bat_feature_cols]

radar_bowl_features = [c for c in ['bowl_economy', 'bowl_dot_pct_bowl', 'bowl_bowling_sr',
                                    'bowl_boundary_concede_pct', 'bowl_pct_powerplay',
                                    'bowl_pct_death', 'bowl_pct_middle'] if c in bowl_feature_cols]

fig, axes = plt.subplots(1, 2, figsize=(14, 7), subplot_kw=dict(polar=True))

# Batting radar
bat_prof = features_train[bat_mask_train].groupby('bat_cluster')[radar_bat_features].mean()
bat_prof_norm = (bat_prof - bat_prof.min()) / (bat_prof.max() - bat_prof.min() + 1e-10)
angles_bat = np.linspace(0, 2 * np.pi, len(radar_bat_features), endpoint=False).tolist()
angles_bat += angles_bat[:1]
colors_r = plt.cm.tab10(np.linspace(0, 1, bat_best_k))

for i, (cid, row) in enumerate(bat_prof_norm.iterrows()):
    vals = row.tolist() + [row.tolist()[0]]
    axes[0].plot(angles_bat, vals, 'o-', linewidth=2, label=f'BAT-{int(cid)}', color=colors_r[i])
    axes[0].fill(angles_bat, vals, alpha=0.08, color=colors_r[i])

axes[0].set_xticks(angles_bat[:-1])
axes[0].set_xticklabels([f.replace('bat_', '').replace('phase_sr_', 'SR_') for f in radar_bat_features], size=8)
axes[0].set_title('Batting Archetypes', size=13, pad=20)
axes[0].legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

# Bowling radar
bowl_prof = features_train[bowl_mask_train].groupby('bowl_cluster')[radar_bowl_features].mean()
bowl_prof_norm = (bowl_prof - bowl_prof.min()) / (bowl_prof.max() - bowl_prof.min() + 1e-10)
angles_bowl = np.linspace(0, 2 * np.pi, len(radar_bowl_features), endpoint=False).tolist()
angles_bowl += angles_bowl[:1]
colors_b = plt.cm.tab10(np.linspace(0, 1, bowl_best_k))

for i, (cid, row) in enumerate(bowl_prof_norm.iterrows()):
    vals = row.tolist() + [row.tolist()[0]]
    axes[1].plot(angles_bowl, vals, 'o-', linewidth=2, label=f'BOWL-{int(cid)}', color=colors_b[i])
    axes[1].fill(angles_bowl, vals, alpha=0.08, color=colors_b[i])

axes[1].set_xticks(angles_bowl[:-1])
axes[1].set_xticklabels([f.replace('bowl_', '').replace('pct_', '') for f in radar_bowl_features], size=8)
axes[1].set_title('Bowling Archetypes', size=13, pad=20)
axes[1].legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cluster_radar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PCA loadings: which features drive the clustering?
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Batting PC1 loadings
bat_loadings = pd.Series(bat_pca.components_[0], index=bat_feature_cols)
top_bat = bat_loadings.abs().nlargest(10)
axes[0].barh(range(len(top_bat)), bat_loadings[top_bat.index].values, color='steelblue')
axes[0].set_yticks(range(len(top_bat)))
axes[0].set_yticklabels([f.replace('bat_', '') for f in top_bat.index], fontsize=9)
axes[0].set_xlabel('PC1 Loading')
axes[0].set_title('Batting: Top Features Driving Separation')

# Bowling PC1 loadings
bowl_loadings = pd.Series(bowl_pca.components_[0], index=bowl_feature_cols)
top_bowl = bowl_loadings.abs().nlargest(10)
axes[1].barh(range(len(top_bowl)), bowl_loadings[top_bowl.index].values, color='firebrick')
axes[1].set_yticks(range(len(top_bowl)))
axes[1].set_yticklabels([f.replace('bowl_', '') for f in top_bowl.index], fontsize=9)
axes[1].set_xlabel('PC1 Loading')
axes[1].set_title('Bowling: Top Features Driving Separation')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pca_loadings.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Playoff vs non-playoff: do playoff teams have HIGHER diversity?
team_bat_entropy_po = team_bat_entropy.merge(
    features_train[['team', 'season', 'is_playoff']].drop_duplicates(),
    on=['team', 'season']
)

po_div = team_bat_entropy_po[team_bat_entropy_po['is_playoff'] == True]['bat_diversity']
npo_div = team_bat_entropy_po[team_bat_entropy_po['is_playoff'] == False]['bat_diversity']
t_stat, p_val = stats.ttest_ind(po_div, npo_div)

print(f"Batting diversity - Playoff: {po_div.mean():.3f}, Non-playoff: {npo_div.mean():.3f}")
print(f"t-test: t={t_stat:.3f}, p={p_val:.4f}")
print(f"{'Playoff teams are significantly MORE diverse' if p_val < 0.05 and t_stat > 0 else 'No significant difference'}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(po_div, bins=15, alpha=0.6, label=f'Playoff (mean={po_div.mean():.2f})', color='green')
ax.hist(npo_div, bins=15, alpha=0.6, label=f'Non-Playoff (mean={npo_div.mean():.2f})', color='red')
ax.set_xlabel('Batting Archetype Diversity (Shannon Entropy)')
ax.set_ylabel('Count')
ax.set_title('Do Playoff Teams Have More Diverse Rosters?')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'diversity_playoff_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Player archetype evolution: how do archetypes shift across seasons?
# Track a few famous players across their careers
career_players = ['V Kohli', 'MS Dhoni', 'RG Sharma', 'DA Warner', 'JJ Bumrah', 'R Ashwin']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, player in enumerate(career_players):
    pdata = features_train[features_train['player'] == player].sort_values('season')
    if len(pdata) == 0:
        continue
    ax = axes[i]

    if pdata['bat_cluster'].notna().any():
        ax.plot(pdata['season'], pdata['bat_cluster'], 'bo-', label='Bat archetype', markersize=6)
    if pdata['bowl_cluster'].notna().any():
        ax.plot(pdata['season'], pdata['bowl_cluster'], 'rs-', label='Bowl archetype', markersize=6)

    ax.set_title(player)
    ax.set_xlabel('Season')
    ax.set_ylabel('Cluster')
    ax.set_yticks(range(max(bat_best_k, bowl_best_k)))
    ax.legend(fontsize=8)

plt.suptitle('Player Archetype Evolution Over Career', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'player_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Replacement recommender validation:
# For players who changed teams 2024->2025, did the new team acquire a similar archetype?
players_2024 = features_train[features_train['season'] == 2024][['player', 'team', 'archetype', 'bat_cluster', 'bowl_cluster']]
players_2025 = features_test[features_test['season'] == 2025][['player', 'team', 'archetype', 'bat_cluster', 'bowl_cluster']]

moved = players_2024.merge(players_2025, on='player', suffixes=('_2024', '_2025'))
moved = moved[moved['team_2024'] != moved['team_2025']]
print(f"Players who changed teams (2024 -> 2025): {len(moved)}")
if len(moved) > 0:
    print(moved[['player', 'team_2024', 'team_2025', 'archetype_2024', 'archetype_2025']].head(15).to_string(index=False))

# Save all outputs
features_train.to_csv(OUTPUT_DIR / 'features_train.csv', index=False)
features_test.to_csv(OUTPUT_DIR / 'features_test.csv', index=False)

bat_profiles[key_bat].to_csv(OUTPUT_DIR / 'batting_archetype_profiles.csv')
bowl_profiles[key_bowl].to_csv(OUTPUT_DIR / 'bowling_archetype_profiles.csv')

print(f"All outputs saved to: {OUTPUT_DIR}")
print(f"\nFigures saved:")
for f in sorted(OUTPUT_DIR.glob('*.png')):
    print(f"  {f.name}")
print(f"\nData saved:")
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    print(f"  {f.name}")

In [ ]:
# Player search + archetype lookup
def search_player(query):
    """Search for players and show their archetypes."""
    all_players = sorted(features_train['player'].unique())
    matches = [p for p in all_players if query.lower() in p.lower()]
    if not matches:
        print(f"No players found matching '{query}'")
        return []

    print(f"Found {len(matches)} match(es) for '{query}':\n")
    for p in matches[:10]:
        latest = features_train[features_train['player'] == p].sort_values('season').iloc[-1]
        bat_c = f"BAT-{int(latest['bat_cluster'])}" if pd.notna(latest.get('bat_cluster')) else ""
        bowl_c = f"BOWL-{int(latest['bowl_cluster'])}" if pd.notna(latest.get('bowl_cluster')) else ""
        archetype = " + ".join(filter(None, [bat_c, bowl_c]))
        seasons = features_train[features_train['player'] == p]['season']
        print(f"  {p} ({latest['player_type']}) | {archetype} | {seasons.min()}-{seasons.max()}")
    return matches

search_player('kohli')
print()
search_player('bumrah')
print()
search_player('jadeja')

## Summary

**Approach:** Cluster batters and bowlers SEPARATELY to discover meaningful subtypes within each role.

**Key outputs:**
1. **Batting archetypes** (k chosen by silhouette): e.g., openers, anchors, finishers, power hitters
2. **Bowling archetypes** (k chosen by silhouette): e.g., death specialists, middle-overs controllers, new-ball strikers
3. **All-rounders** get dual labels (one batting + one bowling archetype)
4. **Team composition analysis**: playoff teams have different archetype mixes than non-playoff teams
5. **Diversity validation**: teams maintain diverse rosters (high entropy) rather than stacking one archetype
6. **Replacement recommender**: find similar players within the same feature space
7. **Temporal validation**: archetypes remain stable on held-out 2025-2026 data